# Mask R-CNN Training for NSFW Detection
## Multi-GPU Training on RTX A2000 (3x 12GB GPUs)

**Dataset**: Balanced dataset with equal class distribution  
**Classes**: anus (0), breast (1), female_genital (2), male_genital (3)  
**Purpose**: Better detection of small areas compared to YOLO


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Hardware: 3x RTX A2000 (12GB VRAM each) | Multi-GPU Training
#  Dataset: final_balanced (4 classes with equal distribution)
#  Model: Mask R-CNN ResNet50 FPN (better for small object detection)
# ═══════════════════════════════════════════════════════════════════════════

import os
import sys
import time
import json
import random
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
from torch.nn.parallel import DataParallel, DistributedDataParallel

import torchvision
from torchvision import transforms
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

import numpy as np
from PIL import Image, ImageFile
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import logging
import builtins

# TensorBoard for logging
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_AVAILABLE = True
except ImportError:
    TENSORBOARD_AVAILABLE = False
    print("⚠ TensorBoard not available. Install: pip install tensorboard")

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Set matplotlib backend
plt.ioff()  # Turn off interactive mode

print("="*100)
print(" Mask R-CNN Training Pipeline - NSFW Content Detection (3x RTX A2000) ".center(100, "="))
print("="*100)
print()

# Check PyTorch and CUDA
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"    Memory: {props.total_memory / (1024**3):.2f} GB")
print()


================= Mask R-CNN Training Pipeline - NSFW Content Detection (4x A100) ==================

PyTorch Version: 2.5.1+cu121
CUDA Available: True
CUDA Version: 12.1
Number of GPUs: 3
  GPU 0: NVIDIA RTX A2000 12GB
    Memory: 11.24 GB
  GPU 1: NVIDIA RTX A2000 12GB
    Memory: 11.24 GB
  GPU 2: NVIDIA RTX A2000 12GB
    Memory: 11.24 GB



## Configuration


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#                            CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

# Dataset paths
BASE_DIR = Path(r"D:\fyp_data")
DATASET_DIR = BASE_DIR / "final_balanced"
DATASET_YAML = DATASET_DIR / "dataset.yaml"

# Training configuration (Optimized for 3x RTX A2000 - 12GB each)
CONFIG = {
    # Model
    'num_classes': 5,  # 4 classes + background (Mask R-CNN requirement)
    'backbone': 'resnet50',  # Change to 'resnet101' for max accuracy (slower but better)
    'pretrained': True,
    
    # Gradient clipping (prevents exploding gradients)
    'grad_clip_max_norm': 10.0,
    
    # Training - OPTIMIZED FOR BEST NSFW DETECTION ACCURACY
    'epochs': 100,  # Increased for better convergence
    'batch_size': 12,  # Total batch size (4 per GPU, faster iterations)
    'num_workers': 2,  # Per GPU (minimal CPU overhead)
    'learning_rate': 0.003,  # Scaled for batch_size=12
    'momentum': 0.9,
    'weight_decay': 0.0005,
    'lr_step_size': 25,  # Drop LR more frequently with larger batches
    'lr_gamma': 0.1,
    
    # Image settings - LARGER FOR BETTER SMALL OBJECT DETECTION
    'min_size': 640,  # Balanced for speed and accuracy
    'max_size': 800,  # Balanced for speed and accuracy
    
    # Multi-GPU - PARALLEL TRAINING
    'use_multi_gpu': True,
    'use_distributed': False,  # False = DataParallel (works in notebook), True = DDP (needs script)
    'gpu_ids': [0, 1, 2],  # All 4 GPUs
    
    # Augmentation
    'horizontal_flip_prob': 0.5,
    
    # Checkpointing
    'save_dir': BASE_DIR / "mask_rcnn_runs",
    'save_period': 5,
    'resume_from': None,  # Path to checkpoint to resume from
    
    # Validation
    'val_period': 3,  # Validate more frequently to catch overfitting earlier
    'eval_metric': 'bbox_map',  # bbox_map or segm_map
    
    # Mixed precision
    'use_amp': True,
    
    # Early stopping - OPTIMIZED
    'early_stopping': True,
    'early_stopping_patience': 15,  # Stop if no improvement for 15 epochs
    'early_stopping_min_delta': 0.0005,  # More sensitive to improvements
    
    # Learning rate warmup - OPTIMIZED
    'warmup_epochs': 5,  # Longer warmup for stability with larger LR
    'warmup_factor': 0.1,  # Start with 10% of base LR
    
    # Grad-CAM visualization
    'gradcam_enabled': True,
    'gradcam_samples': 4,  # Number of samples to visualize per validation
    'gradcam_save_dir': BASE_DIR / "mask_rcnn_runs" / "gradcam",
    
    # TensorBoard logging
    'tensorboard_enabled': True,
    'tensorboard_log_dir': BASE_DIR / "mask_rcnn_runs" / "tensorboard",
    
    # Evaluation metrics
    'compute_map': True,  # Compute mean Average Precision
    'save_confusion_matrix': True,
    'save_predictions': True,
}

# Class names (from YAML)
CLASS_NAMES = {
    0: 'anus',
    1: 'breast',
    2: 'female_genital',
    3: 'male_genital'
}

# Create save directory
CONFIG['save_dir'].mkdir(parents=True, exist_ok=True)

# Setup structured logging (prints still go to notebook + log file)
LOG_DIR = CONFIG['save_dir'] / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_file = LOG_DIR / f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("mask_rcnn")
logger.setLevel(logging.INFO)
logger.propagate = False
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
file_handler = logging.FileHandler(log_file, encoding='utf-8')
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

if hasattr(builtins.print, "_is_logging_wrapper"):
    _original_print = builtins.print._original_print
else:
    _original_print = builtins.print


def print_and_log(*args, **kwargs):
    message = " ".join(str(arg) for arg in args)
    if message.strip():
        logger.info(message)
    _original_print(*args, **kwargs)

print_and_log._is_logging_wrapper = True
print_and_log._original_print = _original_print
builtins.print = print_and_log
print(f"Logging to: {log_file}")

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print()


Configuration:
  num_classes: 5
  backbone: resnet50
  pretrained: True
  grad_clip_max_norm: 10.0
  epochs: 100
  batch_size: 20
  num_workers: 8
  learning_rate: 0.005
  momentum: 0.9
  weight_decay: 0.0005
  lr_step_size: 25
  lr_gamma: 0.1
  min_size: 800
  max_size: 1024
  use_multi_gpu: True
  use_distributed: False
  gpu_ids: [0, 1, 2, 3]
  horizontal_flip_prob: 0.5
  save_dir: D:\fyp_data\mask_rcnn_runs
  save_period: 5
  resume_from: None
  val_period: 3
  eval_metric: bbox_map
  use_amp: True
  early_stopping: True
  early_stopping_patience: 15
  early_stopping_min_delta: 0.0005
  warmup_epochs: 5
  warmup_factor: 0.1
  gradcam_enabled: True
  gradcam_samples: 4
  gradcam_save_dir: D:\fyp_data\mask_rcnn_runs\gradcam
  tensorboard_enabled: True
  tensorboard_log_dir: D:\fyp_data\mask_rcnn_runs\tensorboard
  compute_map: True
  save_confusion_matrix: True
  save_predictions: True



## Dataset Class (YOLO to COCO format converter)


In [ ]:
class YOLOToMaskRCNNDataset(Dataset):
    """
    Converts YOLO segmentation format dataset to Mask R-CNN format.
    
    YOLO format: class_id x1 y1 x2 y2 ... (polygon segmentation, normalized 0-1)
    Mask R-CNN needs: boxes (x1, y1, x2, y2 in pixels), labels
    
    Class Mapping (NSFW Detection):
    - YOLO 0 (anus) -> Mask R-CNN 1 (anus)
    - YOLO 1 (breast) -> Mask R-CNN 2 (breast) 
    - YOLO 2 (female_genital) -> Mask R-CNN 3 (female_genital)
    - YOLO 3 (male_genital) -> Mask R-CNN 4 (male_genital)
    - Mask R-CNN 0 = background (reserved by framework)
    """
    
    def __init__(self, dataset_dir, split='train', transform=None, min_size=640, max_size=800):
        self.dataset_dir = Path(dataset_dir)
        self.split = split
        self.transform = transform
        self.min_size = min_size
        self.max_size = max_size
        
        # Paths
        self.images_dir = self.dataset_dir / "images" / split
        self.labels_dir = self.dataset_dir / "labels" / split
        
        # Get all image files
        self.image_files = []
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            self.image_files.extend(list(self.images_dir.glob(f"*{ext}")))
        
        self.image_files = sorted(self.image_files)
        print(f"Found {len(self.image_files)} images in {split} split")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        image = np.array(image)
        original_height, original_width = image.shape[:2]
        
        # Load labels (YOLO format)
        label_path = self.labels_dir / f"{img_path.stem}.txt"
        boxes = []
        labels = []
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split()
                if len(parts) < 5:
                    continue
                
                try:
                    class_id = int(parts[0])
                    
                    # YOLO segmentation format: class_id x1 y1 x2 y2 ... (polygon points, normalized)
                    # We need to extract bounding box from polygon
                    coords = [float(x) for x in parts[1:]]
                    
                    if len(coords) < 4:  # Need at least 2 points (x, y pairs)
                        continue
                    
                    # Extract x and y coordinates from polygon
                    xs = [coords[i] for i in range(0, len(coords), 2)]  # Even indices
                    ys = [coords[i] for i in range(1, len(coords), 2)]  # Odd indices
                    
                    # Get bounding box from polygon (normalized coordinates)
                    x_min_norm = min(xs)
                    y_min_norm = min(ys)
                    x_max_norm = max(xs)
                    y_max_norm = max(ys)
                    
                    # Convert to absolute coordinates
                    x1 = x_min_norm * original_width
                    y1 = y_min_norm * original_height
                    x2 = x_max_norm * original_width
                    y2 = y_max_norm * original_height
                    
                    # Ensure valid box
                    x1 = max(0, min(x1, original_width))
                    y1 = max(0, min(y1, original_height))
                    x2 = max(0, min(x2, original_width))
                    y2 = max(0, min(y2, original_height))
                    
                    if x2 > x1 and y2 > y1:  # Valid box
                        boxes.append([x1, y1, x2, y2])
                        labels.append(class_id + 1)  # +1 because 0 is background in Mask R-CNN
                
                except (ValueError, IndexError):
                    continue
        
        # Convert boxes/labels to numpy for potential scaling
        if len(boxes) == 0:
            boxes_np = np.zeros((0, 4), dtype=np.float32)
            labels_np = np.zeros((0,), dtype=np.int64)
        else:
            boxes_np = np.array(boxes, dtype=np.float32)
            labels_np = np.array(labels, dtype=np.int64)
        
        # Resize image to stay within [min_size, max_size] for faster loading
        min_dim = min(original_height, original_width)
        max_dim = max(original_height, original_width)
        scale = 1.0
        if min_dim > 0 and max_dim > 0:
            scale = self.min_size / min_dim
            if scale * max_dim > self.max_size:
                scale = self.max_size / max_dim
        if abs(scale - 1.0) > 1e-3:
            new_width = int(round(original_width * scale))
            new_height = int(round(original_height * scale))
            image = np.array(Image.fromarray(image).resize((new_width, new_height), Image.BILINEAR))
            if len(boxes_np) > 0:
                boxes_np *= scale
            original_height, original_width = new_height, new_width
        
        # Convert image and labels to tensors
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        boxes = torch.from_numpy(boxes_np)
        labels = torch.from_numpy(labels_np)
        
        # Create target dictionary (Mask R-CNN format)
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros((0,)),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64) if len(boxes) > 0 else torch.zeros((0,)),
        }
        
        # Apply transforms if provided
        if self.transform:
            image, target = self.transform(image, target)
        
        return image, target


def collate_fn(batch):
    """
    Custom collate function for batching.
    Mask R-CNN needs images and targets as separate lists.
    """
    images = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    return images, targets


# Test dataset loading
print("Testing dataset loading...")
try:
    test_dataset = YOLOToMaskRCNNDataset(DATASET_DIR, split='train', min_size=CONFIG['min_size'], max_size=CONFIG['max_size'])
    if len(test_dataset) > 0:
        sample_img, sample_target = test_dataset[0]
        print(f"✓ Dataset loaded successfully")
        print(f"  Sample image shape: {sample_img.shape}")
        print(f"  Sample target boxes: {len(sample_target['boxes'])}")
        print(f"  Sample target labels: {sample_target['labels']}")
    else:
        print("⚠ Dataset is empty - waiting for dataset creation to complete...")
except Exception as e:
    print(f"⚠ Dataset not ready yet: {e}")
    print("  This is expected if dataset creation is still running in background")
print()


Testing dataset loading...
Found 63856 images in train split
✓ Dataset loaded successfully
  Sample image shape: torch.Size([3, 2208, 1242])
  Sample target boxes: 2
  Sample target labels: tensor([1, 3])

Found 63856 images in train split
✓ Dataset loaded successfully
  Sample image shape: torch.Size([3, 2208, 1242])
  Sample target boxes: 2
  Sample target labels: tensor([1, 3])



## Model Setup


In [8]:
def get_model(num_classes, pretrained=True, backbone='resnet50'):
    """
    Create Mask R-CNN model with ResNet50 or ResNet101 backbone.
    num_classes includes background (so 5 for 4 classes).
    
    Args:
        num_classes: Number of classes (including background)
        pretrained: Use pretrained weights
        backbone: 'resnet50' or 'resnet101'
    """
    if backbone == 'resnet101':
        # Use ResNet101 backbone for better accuracy
        from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
        from torchvision.models.detection import MaskRCNN
        
        # Create ResNet101 FPN backbone
        backbone_net = resnet_fpn_backbone('resnet101', pretrained=pretrained)
        
        # Create Mask R-CNN with custom backbone
        model = MaskRCNN(
            backbone=backbone_net,
            num_classes=num_classes,
            min_size=CONFIG['min_size'],
            max_size=CONFIG['max_size']
        )
        print("✓ Using ResNet101 backbone (deeper network, better accuracy)")
    else:
        # Default: ResNet50
        model = maskrcnn_resnet50_fpn(pretrained=pretrained)
        print("✓ Using ResNet50 backbone (faster training)")
    
    # Replace the classifier heads for our number of classes
    # Get number of input features
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # Replace box predictor
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Replace mask predictor
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )
    
    return model


# Initialize model
print("Initializing Mask R-CNN model...")
model = get_model(CONFIG['num_classes'], pretrained=CONFIG['pretrained'], backbone=CONFIG['backbone'])
print(f"✓ Model created with {CONFIG['num_classes']} classes")
print(f"  Backbone: {CONFIG['backbone']}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print()


Initializing Mask R-CNN model...
✓ Using ResNet50 backbone (faster training)
✓ Model created with 5 classes
  Backbone: resnet50
  Total parameters: 43,938,541
  Trainable parameters: 43,716,141

✓ Using ResNet50 backbone (faster training)
✓ Model created with 5 classes
  Backbone: resnet50
  Total parameters: 43,938,541
  Trainable parameters: 43,716,141



## Multi-GPU Setup


In [9]:
def setup_multi_gpu(model, config):
    """
    Setup model for multi-GPU training.
    Returns model, device, and whether to use DataParallel or DistributedDataParallel.
    """
    if not torch.cuda.is_available():
        print("⚠ CUDA not available, using CPU")
        return model.to('cpu'), 'cpu', False
    
    num_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {num_gpus}")
    
    if config['use_multi_gpu'] and num_gpus > 1:
        if config['use_distributed']:
            # DistributedDataParallel (better for large scale)
            # Note: This requires running with torch.distributed.launch
            print("Using DistributedDataParallel (requires torch.distributed.launch)")
            # This would be set up in a separate training script
            device = torch.device('cuda')
            model = model.to(device)
            # DDP setup would happen here
            return model, device, True
        else:
            # DataParallel (simpler, works in notebook)
            print(f"Using DataParallel on {num_gpus} GPUs")
            device = torch.device('cuda:0')
            model = model.to(device)
            model = DataParallel(model, device_ids=config['gpu_ids'][:num_gpus])
            return model, device, False
    else:
        # Single GPU
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        print(f"Using single device: {device}")
        return model, device, False


# Setup model for multi-GPU
model, device, use_ddp = setup_multi_gpu(model, CONFIG)
print(f"✓ Model moved to device: {device}")
print()


Available GPUs: 3
Using DataParallel on 3 GPUs
✓ Model moved to device: cuda:0



## Data Loaders


In [10]:
# Create datasets
print("Creating data loaders...")
train_dataset = YOLOToMaskRCNNDataset(
    DATASET_DIR, 
    split='train',
    min_size=CONFIG['min_size'],
    max_size=CONFIG['max_size']
)

val_dataset = YOLOToMaskRCNNDataset(
    DATASET_DIR,
    split='val',
    min_size=CONFIG['min_size'],
    max_size=CONFIG['max_size']
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✓ Train loader: {len(train_loader)} batches")
print(f"✓ Val loader: {len(val_loader)} batches")
print()


Creating data loaders...
Found 63856 images in train split
Found 8026 images in val split
✓ Train loader: 3193 batches
✓ Val loader: 402 batches

Found 63856 images in train split
Found 8026 images in val split
✓ Train loader: 3193 batches
✓ Val loader: 402 batches



In [ ]:
import time

print("Testing if workers can load even 1 batch...")
print(f"Current num_workers: {CONFIG['num_workers']}")
print("Attempting to load first batch (will timeout after 60 seconds)...\n")

start = time.time()
timeout = 60  # 60 second timeout

try:
    # Try to get first batch with timeout
    train_iter = iter(train_loader)
    
    # This is where it will hang
    print("Calling next(train_iter)... (this is where it usually hangs)")
    batch = next(train_iter)
    
    elapsed = time.time() - start
    print(f"\n✓ SUCCESS! Loaded first batch in {elapsed:.1f} seconds")
    print(f"  Images in batch: {len(batch[0])}")
    
except KeyboardInterrupt:
    print(f"\n❌ HUNG! Waited {time.time() - start:.1f} seconds, manually stopped")
    print("  This confirms: Workers are deadlocked")
    
except Exception as e:
    print(f"\n❌ ERROR after {time.time() - start:.1f} seconds: {e}")

## Training Setup (Optimizer, Scheduler, Loss)


In [11]:
# Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(
    params,
    lr=CONFIG['learning_rate'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay']
)

# Learning rate scheduler
lr_scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=CONFIG['lr_step_size'],
    gamma=CONFIG['lr_gamma']
)

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler() if CONFIG['use_amp'] else None

print("✓ Optimizer: SGD")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Momentum: {CONFIG['momentum']}")
print(f"  Weight decay: {CONFIG['weight_decay']}")
print(f"✓ Scheduler: StepLR (step_size={CONFIG['lr_step_size']}, gamma={CONFIG['lr_gamma']})")
if CONFIG['use_amp']:
    print("✓ Mixed precision training enabled")
print()


✓ Optimizer: SGD
  Learning rate: 0.005
  Momentum: 0.9
  Weight decay: 0.0005
✓ Scheduler: StepLR (step_size=25, gamma=0.1)
✓ Mixed precision training enabled



## Advanced Features: Grad-CAM, mAP, TensorBoard, Early Stopping


In [12]:
# ═════════════════════════════════════════════════════════════════════════════
#                    GRAD-CAM VISUALIZATION FOR MASK R-CNN
# ═════════════════════════════════════════════════════════════════════════════

class GradCAM:
    """
    Gradient-weighted Class Activation Mapping for Mask R-CNN.
    Shows which regions the model focuses on for predictions.
    """
    
    def __init__(self, model, target_layer_name='backbone.body.layer4'):
        self.model = model
        self.target_layer = None
        self.gradients = None
        self.activations = None
        
        # Hook to get activations and gradients
        self.hooks = []
        self._register_hooks(target_layer_name)
    
    def _register_hooks(self, layer_name):
        """Register forward and backward hooks."""
        def forward_hook(module, input, output):
            self.activations = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        # Find the target layer in the model
        for name, module in self.model.named_modules():
            if layer_name in name:
                self.target_layer = module
                self.hooks.append(module.register_forward_hook(forward_hook))
                self.hooks.append(module.register_backward_hook(backward_hook))
                break
        
        if self.target_layer is None:
            # Fallback to backbone's last layer
            backbone = self.model.backbone.body
            self.target_layer = backbone.layer4
            self.hooks.append(self.target_layer.register_forward_hook(forward_hook))
            self.hooks.append(self.target_layer.register_backward_hook(backward_hook))
    
    def generate_cam(self, image, target_class=None):
        """
        Generate Grad-CAM heatmap for an image.
        
        Args:
            image: Input image tensor [C, H, W]
            target_class: Class ID to visualize (None = use highest score)
        
        Returns:
            heatmap: Grad-CAM heatmap as numpy array
        """
        self.model.eval()
        self.gradients = None
        self.activations = None
        
        # Forward pass
        image_batch = [image.to(next(self.model.parameters()).device)]
        predictions = self.model(image_batch)
        
        if len(predictions) == 0 or len(predictions[0]['boxes']) == 0:
            return None
        
        # Get the highest scoring box or target class
        scores = predictions[0]['scores']
        labels = predictions[0]['labels']
        
        if target_class is None:
            # Use highest scoring detection
            max_idx = scores.argmax().item()
            target_class = labels[max_idx].item()
            target_score = scores[max_idx].item()
        else:
            # Find box with target class
            mask = labels == target_class
            if not mask.any():
                return None
            max_idx = scores[mask].argmax().item()
            target_score = scores[mask][max_idx].item()
        
        # Backward pass
        self.model.zero_grad()
        target_score.backward(retain_graph=True)
        
        if self.gradients is None or self.activations is None:
            return None
        
        # Compute Grad-CAM
        gradients = self.gradients[0]  # [B, C, H, W]
        activations = self.activations  # [B, C, H, W]
        
        # Global average pooling of gradients
        weights = gradients.mean(dim=(2, 3), keepdim=True)  # [B, C, 1, 1]
        
        # Weighted combination of activation maps
        cam = (weights * activations).sum(dim=1, keepdim=True)  # [B, 1, H, W]
        cam = torch.relu(cam)  # ReLU to get positive activations
        
        # Normalize
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam, target_class, target_score
    
    def visualize(self, image, original_image, target_class=None, save_path=None):
        """
        Visualize Grad-CAM on image.
        
        Args:
            image: Preprocessed image tensor
            original_image: Original PIL image
            target_class: Class to visualize
            save_path: Path to save visualization
        """
        cam, pred_class, score = self.generate_cam(image, target_class)
        
        if cam is None:
            return None
        
        # Resize CAM to original image size
        from torchvision.transforms import functional as F
        cam_resized = F.resize(
            torch.from_numpy(cam).unsqueeze(0),
            original_image.size[::-1]  # (H, W)
        ).squeeze().numpy()
        
        # Create visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original image
        axes[0].imshow(original_image)
        axes[0].set_title('Original Image', fontsize=12)
        axes[0].axis('off')
        
        # Heatmap
        im = axes[1].imshow(cam_resized, cmap='jet', alpha=0.5)
        axes[1].set_title(f'Grad-CAM Heatmap\\nClass: {CLASS_NAMES.get(pred_class-1, pred_class)}, Score: {score:.3f}', fontsize=12)
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1])
        
        # Overlay
        axes[2].imshow(original_image)
        axes[2].imshow(cam_resized, cmap='jet', alpha=0.4)
        axes[2].set_title('Overlay', fontsize=12)
        axes[2].axis('off')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close()
        else:
            plt.show()
        
        return fig

print("✓ Grad-CAM class ready")
print()


✓ Grad-CAM class ready





In [13]:
# ═════════════════════════════════════════════════════════════════════════════
#                    MEAN AVERAGE PRECISION (mAP) CALCULATION
# ═════════════════════════════════════════════════════════════════════════════

def calculate_map(predictions, targets, iou_threshold=0.5):
    """
    Calculate mean Average Precision (mAP) for object detection.
    
    Args:
        predictions: List of prediction dicts from model
        targets: List of target dicts from dataset
        iou_threshold: IoU threshold for positive detection
    
    Returns:
        map_score: Mean Average Precision
        ap_per_class: AP for each class
    """
    from torchvision.ops import box_iou
    
    all_detections = {i: [] for i in range(1, CONFIG['num_classes'])}  # Class 1-4
    all_ground_truths = {i: [] for i in range(1, CONFIG['num_classes'])}
    
    # Collect all detections and ground truths
    for pred, target in zip(predictions, targets):
        pred_boxes = pred['boxes'].cpu()
        pred_labels = pred['labels'].cpu()
        pred_scores = pred['scores'].cpu()
        
        gt_boxes = target['boxes'].cpu()
        gt_labels = target['labels'].cpu()
        
        # Group by class
        for class_id in range(1, CONFIG['num_classes']):
            # Predictions for this class
            pred_mask = pred_labels == class_id
            if pred_mask.any():
                all_detections[class_id].append({
                    'boxes': pred_boxes[pred_mask],
                    'scores': pred_scores[pred_mask]
                })
            else:
                all_detections[class_id].append({
                    'boxes': torch.empty((0, 4)),
                    'scores': torch.empty((0,))
                })
            
            # Ground truths for this class
            gt_mask = gt_labels == class_id
            if gt_mask.any():
                all_ground_truths[class_id].append(gt_boxes[gt_mask])
            else:
                all_ground_truths[class_id].append(torch.empty((0, 4)))
    
    # Calculate AP for each class
    ap_per_class = {}
    for class_id in range(1, CONFIG['num_classes']):
        class_detections = all_detections[class_id]
        class_ground_truths = all_ground_truths[class_id]
        
        # Flatten all detections and ground truths
        all_pred_boxes = []
        all_pred_scores = []
        all_gt_boxes = []
        
        for det, gt in zip(class_detections, class_ground_truths):
            all_pred_boxes.append(det['boxes'])
            all_pred_scores.append(det['scores'])
            all_gt_boxes.append(gt)
        
        if len(all_pred_boxes) == 0:
            ap_per_class[class_id] = 0.0
            continue
        
        # Concatenate
        pred_boxes = torch.cat(all_pred_boxes, dim=0)
        pred_scores = torch.cat(all_pred_scores, dim=0)
        gt_boxes = torch.cat(all_gt_boxes, dim=0)
        
        if len(pred_boxes) == 0 or len(gt_boxes) == 0:
            ap_per_class[class_id] = 0.0
            continue
        
        # Sort by score
        sorted_indices = pred_scores.argsort(descending=True)
        pred_boxes = pred_boxes[sorted_indices]
        pred_scores = pred_scores[sorted_indices]
        
        # Calculate precision and recall
        tp = torch.zeros(len(pred_boxes))
        fp = torch.zeros(len(pred_boxes))
        gt_matched = torch.zeros(len(gt_boxes), dtype=torch.bool)
        
        for i, pred_box in enumerate(pred_boxes):
            # Calculate IoU with all ground truths
            ious = box_iou(pred_box.unsqueeze(0), gt_boxes)
            max_iou, max_idx = ious.max(dim=1)
            
            if max_iou.item() >= iou_threshold and not gt_matched[max_idx]:
                tp[i] = 1
                gt_matched[max_idx] = True
            else:
                fp[i] = 1
        
        # Calculate precision and recall
        tp_cumsum = tp.cumsum(0)
        fp_cumsum = fp.cumsum(0)
        
        recalls = tp_cumsum / len(gt_boxes) if len(gt_boxes) > 0 else torch.zeros_like(tp_cumsum)
        precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-8)
        
        # Calculate AP using 11-point interpolation
        ap = 0.0
        for t in torch.arange(0, 1.1, 0.1):
            mask = recalls >= t
            if mask.any():
                p = precisions[mask].max()
                ap += p / 11.0
        
        ap_per_class[class_id] = ap.item()
    
    # Calculate mAP
    map_score = sum(ap_per_class.values()) / len(ap_per_class)
    
    return map_score, ap_per_class

print("✓ mAP calculation function ready")
print()


✓ mAP calculation function ready



In [14]:
# ═════════════════════════════════════════════════════════════════════════════
#                    TENSORBOARD SETUP
# ═════════════════════════════════════════════════════════════════════════════

# Initialize TensorBoard writer
if CONFIG['tensorboard_enabled'] and TENSORBOARD_AVAILABLE:
    CONFIG['tensorboard_log_dir'].mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(str(CONFIG['tensorboard_log_dir']))
    print(f"✓ TensorBoard logging enabled")
    print(f"  Log directory: {CONFIG['tensorboard_log_dir']}")
    print(f"  View with: tensorboard --logdir {CONFIG['tensorboard_log_dir']}")
else:
    writer = None
    if not TENSORBOARD_AVAILABLE:
        print("⚠ TensorBoard not available. Install: pip install tensorboard")
    else:
        print("⚠ TensorBoard disabled in config")

# Create Grad-CAM directory
if CONFIG['gradcam_enabled']:
    CONFIG['gradcam_save_dir'].mkdir(parents=True, exist_ok=True)
    print(f"✓ Grad-CAM visualization enabled")
    print(f"  Save directory: {CONFIG['gradcam_save_dir']}")
print()


✓ TensorBoard logging enabled
  Log directory: D:\fyp_data\mask_rcnn_runs\tensorboard
  View with: tensorboard --logdir D:\fyp_data\mask_rcnn_runs\tensorboard
✓ Grad-CAM visualization enabled
  Save directory: D:\fyp_data\mask_rcnn_runs\gradcam



## Additional Production Features: Confusion Matrix, Prediction Visualization, LR Finder, Model Export


In [15]:
# ═════════════════════════════════════════════════════════════════════════════
#                    CONFUSION MATRIX & PER-CLASS METRICS
# ═════════════════════════════════════════════════════════════════════════════

def calculate_per_class_metrics(predictions, targets, iou_threshold=0.5, conf_threshold=0.5):
    """
    Calculate precision, recall, and F1 score per class.
    
    Returns:
        metrics: Dict with per-class precision, recall, F1
        confusion_matrix: Confusion matrix array
    """
    from torchvision.ops import box_iou
    
    # Initialize counters
    tp_per_class = {i: 0 for i in range(1, CONFIG['num_classes'])}
    fp_per_class = {i: 0 for i in range(1, CONFIG['num_classes'])}
    fn_per_class = {i: 0 for i in range(1, CONFIG['num_classes'])}
    
    # Process each image
    for pred, target in zip(predictions, targets):
        pred_boxes = pred['boxes'].cpu()
        pred_labels = pred['labels'].cpu()
        pred_scores = pred['scores'].cpu()
        
        gt_boxes = target['boxes'].cpu()
        gt_labels = target['labels'].cpu()
        
        # Filter by confidence
        conf_mask = pred_scores >= conf_threshold
        pred_boxes = pred_boxes[conf_mask]
        pred_labels = pred_labels[conf_mask]
        pred_scores = pred_scores[conf_mask]
        
        # Match predictions to ground truth
        if len(gt_boxes) > 0 and len(pred_boxes) > 0:
            ious = box_iou(pred_boxes, gt_boxes)
            
            # Match each prediction to best ground truth
            for i, (pred_box, pred_label, pred_score) in enumerate(zip(pred_boxes, pred_labels, pred_scores)):
                # Find best matching ground truth
                best_iou, best_gt_idx = ious[i].max(dim=0)
                
                if best_iou >= iou_threshold:
                    gt_label = gt_labels[best_gt_idx].item()
                    if pred_label.item() == gt_label:
                        tp_per_class[pred_label.item()] += 1
                    else:
                        fp_per_class[pred_label.item()] += 1
                        fn_per_class[gt_label] += 1
                    # Mark as matched
                    ious[:, best_gt_idx] = -1
                else:
                    fp_per_class[pred_label.item()] += 1
        
        # Count unmatched ground truths as false negatives
        for gt_label in gt_labels:
            fn_per_class[gt_label.item()] += 1
    
    # Calculate metrics per class
    metrics = {}
    for class_id in range(1, CONFIG['num_classes']):
        tp = tp_per_class[class_id]
        fp = fp_per_class[class_id]
        fn = fn_per_class[class_id]
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
        
        metrics[class_id] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'fp': fp,
            'fn': fn
        }
    
    # Build confusion matrix (simplified - just for visualization)
    # This is a simplified version showing class predictions
    cm_data = []
    all_pred_labels = []
    all_gt_labels = []
    
    for pred, target in zip(predictions, targets):
        pred_labels = pred['labels'].cpu()
        pred_scores = pred['scores'].cpu()
        gt_labels = target['labels'].cpu()
        
        # Filter by confidence
        conf_mask = pred_scores >= conf_threshold
        pred_labels = pred_labels[conf_mask]
        
        all_pred_labels.extend(pred_labels.tolist())
        all_gt_labels.extend(gt_labels.tolist())
    
    # Create confusion matrix
    if len(all_pred_labels) > 0 and len(all_gt_labels) > 0:
        # Pad to same length (simplified approach)
        max_len = max(len(all_pred_labels), len(all_gt_labels))
        pred_padded = all_pred_labels + [0] * (max_len - len(all_pred_labels))
        gt_padded = all_gt_labels + [0] * (max_len - len(all_gt_labels))
        
        cm = confusion_matrix(gt_padded[:len(all_gt_labels)], 
                             all_pred_labels[:len(all_gt_labels)],
                             labels=list(range(1, CONFIG['num_classes'])))
    else:
        cm = np.zeros((CONFIG['num_classes']-1, CONFIG['num_classes']-1))
    
    return metrics, cm


def plot_confusion_matrix(cm, epoch, save_path=None):
    """Plot and save confusion matrix."""
    class_names = [CLASS_NAMES[i] for i in range(4)]  # 4 classes
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

print("✓ Confusion matrix and per-class metrics functions ready")
print()


✓ Confusion matrix and per-class metrics functions ready



In [16]:
# ═════════════════════════════════════════════════════════════════════════════
#                    PREDICTION VISUALIZATION
# ═════════════════════════════════════════════════════════════════════════════

def visualize_predictions(image, predictions, targets=None, save_path=None, conf_threshold=0.5):
    """
    Visualize model predictions with bounding boxes.
    
    Args:
        image: PIL Image or numpy array
        predictions: Model prediction dict
        targets: Ground truth dict (optional)
        save_path: Path to save visualization
        conf_threshold: Confidence threshold for display
    """
    # Convert image to numpy if PIL
    if isinstance(image, Image.Image):
        img_array = np.array(image)
    else:
        img_array = image
    
    fig, axes = plt.subplots(1, 2 if targets else 1, figsize=(15, 7) if targets else (10, 7))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    
    # Color map for classes
    colors = {
        1: 'red',      # anus
        2: 'blue',     # breast
        3: 'green',    # female_genital
        4: 'orange'    # male_genital
    }
    
    # Plot predictions
    ax = axes[0]
    ax.imshow(img_array)
    ax.set_title('Predictions', fontsize=14, fontweight='bold')
    
    pred_boxes = predictions['boxes'].cpu()
    pred_labels = predictions['labels'].cpu()
    pred_scores = predictions['scores'].cpu()
    
    # Filter by confidence
    conf_mask = pred_scores >= conf_threshold
    pred_boxes = pred_boxes[conf_mask]
    pred_labels = pred_labels[conf_mask]
    pred_scores = pred_scores[conf_mask]
    
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        x1, y1, x2, y2 = box.tolist()
        color = colors.get(label.item(), 'yellow')
        class_name = CLASS_NAMES.get(label.item() - 1, f"class_{label.item()}")
        
        # Draw bounding box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Add label
        ax.text(x1, y1-5, f"{class_name} {score:.2f}", 
               color=color, fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    ax.axis('off')
    
    # Plot ground truth if available
    if targets is not None:
        ax = axes[1]
        ax.imshow(img_array)
        ax.set_title('Ground Truth', fontsize=14, fontweight='bold')
        
        gt_boxes = targets['boxes'].cpu()
        gt_labels = targets['labels'].cpu()
        
        for box, label in zip(gt_boxes, gt_labels):
            x1, y1, x2, y2 = box.tolist()
            color = colors.get(label.item(), 'yellow')
            class_name = CLASS_NAMES.get(label.item() - 1, f"class_{label.item()}")
            
            # Draw bounding box
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                    linewidth=2, edgecolor=color, facecolor='none', linestyle='--')
            ax.add_patch(rect)
            
            # Add label
            ax.text(x1, y1-5, class_name, 
                   color=color, fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
        
        ax.axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

print("✓ Prediction visualization function ready")
print()


✓ Prediction visualization function ready



In [17]:
# ═════════════════════════════════════════════════════════════════════════════
#                    LEARNING RATE FINDER
# ═════════════════════════════════════════════════════════════════════════════

def find_learning_rate(model, train_loader, device, init_lr=1e-8, final_lr=10, num_iter=100):
    """
    Find optimal learning rate using LR range test.
    
    Returns:
        best_lr: Recommended learning rate
        lrs: List of tested learning rates
        losses: List of corresponding losses
    """
    print("Finding optimal learning rate...")
    print("This will run a few iterations to find the best LR.")
    
    # Save original state
    original_state = model.state_dict()
    original_optimizer_state = optimizer.state_dict()
    
    # Create new optimizer for LR test
    test_optimizer = optim.SGD(model.parameters(), lr=init_lr)
    
    lrs = []
    losses = []
    best_loss = float('inf')
    best_lr = init_lr
    
    # Exponential range
    lr_mult = (final_lr / init_lr) ** (1 / num_iter)
    
    model.train()
    iterator = iter(train_loader)
    
    for i in range(num_iter):
        try:
            images, targets = next(iterator)
        except StopIteration:
            iterator = iter(train_loader)
            images, targets = next(iterator)
        
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Update LR
        current_lr = init_lr * (lr_mult ** i)
        for param_group in test_optimizer.param_groups:
            param_group['lr'] = current_lr
        
        # Forward and backward
        test_optimizer.zero_grad()
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        test_optimizer.step()
        
        lrs.append(current_lr)
        losses.append(loss.item())
        
        # Track best (lowest) loss
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_lr = current_lr
        
        if (i + 1) % 20 == 0:
            print(f"  Iteration {i+1}/{num_iter}: LR={current_lr:.2e}, Loss={loss.item():.4f}")
    
    # Restore original state
    model.load_state_dict(original_state)
    optimizer.load_state_dict(original_optimizer_state)
    
    # Find LR where loss decreases fastest (steepest point)
    # Use smoothed loss to find steepest descent
    if len(losses) > 10:
        smoothed_losses = []
        window = 5
        for i in range(len(losses)):
            start = max(0, i - window)
            end = min(len(losses), i + window + 1)
            smoothed_losses.append(np.mean(losses[start:end]))
        
        # Find point of steepest descent
        gradients = np.gradient(smoothed_losses)
        steepest_idx = np.argmin(gradients)  # Most negative gradient
        recommended_lr = lrs[steepest_idx]
    else:
        recommended_lr = best_lr
    
    print(f"\\n✓ LR Finder Complete")
    print(f"  Recommended LR: {recommended_lr:.6f}")
    print(f"  Best LR (lowest loss): {best_lr:.6f}")
    print(f"  Note: You may want to use 0.1-0.5x of recommended LR for stability")
    
    return recommended_lr, lrs, losses

print("✓ Learning rate finder function ready")
print()


✓ Learning rate finder function ready



In [18]:
# ═════════════════════════════════════════════════════════════════════════════
#                    MODEL EXPORT (ONNX/TorchScript)
# ═════════════════════════════════════════════════════════════════════════════

def export_model(model, save_dir, format='onnx'):
    """
    Export trained model for deployment.
    
    Args:
        model: Trained model
        save_dir: Directory to save exported model
        format: 'onnx' or 'torchscript'
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Get model without DataParallel wrapper
    model_for_export = model.module if isinstance(model, (DataParallel, DistributedDataParallel)) else model
    model_for_export.eval()
    
    # Create dummy input
    dummy_input = torch.randn(1, 3, 640, 640).to(next(model_for_export.parameters()).device)
    
    if format.lower() == 'onnx':
        try:
            save_path = save_dir / "model.onnx"
            torch.onnx.export(
                model_for_export,
                [dummy_input],
                str(save_path),
                input_names=['image'],
                output_names=['boxes', 'labels', 'scores'],
                dynamic_axes={
                    'image': {0: 'batch_size'},
                    'boxes': {0: 'num_detections'},
                    'labels': {0: 'num_detections'},
                    'scores': {0: 'num_detections'}
                },
                opset_version=11
            )
            print(f"✓ Model exported to ONNX: {save_path}")
            return save_path
        except Exception as e:
            print(f"⚠ ONNX export failed: {e}")
            print("  Trying TorchScript instead...")
            format = 'torchscript'
    
    if format.lower() == 'torchscript':
        try:
            # Trace the model
            traced_model = torch.jit.trace(model_for_export, [dummy_input])
            save_path = save_dir / "model.pt"
            traced_model.save(str(save_path))
            print(f"✓ Model exported to TorchScript: {save_path}")
            return save_path
        except Exception as e:
            print(f"⚠ TorchScript export failed: {e}")
            return None
    
    return None

print("✓ Model export function ready")
print()


✓ Model export function ready



## Comprehensive Charts & Visualizations

Generate and save all training curves, metrics charts, and analysis plots.


In [19]:
# ═════════════════════════════════════════════════════════════════════════════
#                    COMPREHENSIVE TRAINING VISUALIZATIONS
# ═════════════════════════════════════════════════════════════════════════════

def plot_training_curves(train_losses, val_losses, map_scores=None, save_path=None):
    """Plot comprehensive training curves."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    epochs = range(1, len(train_losses) + 1)
    
    # 1. Training and Validation Loss
    ax = axes[0, 0]
    ax.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    if val_losses:
        val_epochs = [i * CONFIG['val_period'] for i in range(1, len(val_losses) + 1)]
        ax.plot(val_epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # 2. mAP Score
    ax = axes[0, 1]
    if map_scores and len(map_scores) > 0:
        map_epochs = [i * CONFIG['val_period'] for i in range(1, len(map_scores) + 1)]
        ax.plot(map_epochs, map_scores, 'g-', marker='o', label='mAP@0.5', linewidth=2, markersize=6)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('mAP', fontsize=12)
        ax.set_title('Mean Average Precision (mAP)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    else:
        ax.text(0.5, 0.5, 'mAP data not available', 
               ha='center', va='center', fontsize=12, transform=ax.transAxes)
        ax.set_title('Mean Average Precision (mAP)', fontsize=14, fontweight='bold')
    
    # 3. Loss Components (if available)
    ax = axes[1, 0]
    # This would show individual loss components if tracked
    ax.text(0.5, 0.5, 'Loss components visualization\\n(available in TensorBoard)', 
           ha='center', va='center', fontsize=12, transform=ax.transAxes)
    ax.set_title('Loss Components', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # 4. Training Progress
    ax = axes[1, 1]
    if val_losses and train_losses:
        # Calculate gap between train and val
        val_epochs_aligned = []
        train_losses_aligned = []
        for i, val_loss in enumerate(val_losses):
            epoch = (i + 1) * CONFIG['val_period']
            if epoch <= len(train_losses):
                val_epochs_aligned.append(epoch)
                train_losses_aligned.append(train_losses[epoch - 1])
        
        if val_epochs_aligned:
            gap = [t - v for t, v in zip(train_losses_aligned, val_losses)]
            ax.plot(val_epochs_aligned, gap, 'purple', marker='s', 
                   label='Train-Val Gap', linewidth=2, markersize=6)
            ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
            ax.set_xlabel('Epoch', fontsize=12)
            ax.set_ylabel('Loss Difference', fontsize=12)
            ax.set_title('Overfitting Indicator (Train - Val Loss)', fontsize=14, fontweight='bold')
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig


def plot_per_class_metrics_history(metrics_history, save_path=None):
    """Plot per-class metrics over time."""
    if not metrics_history:
        return None
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Extract data
    epochs = []
    precision_data = {i: [] for i in range(1, CONFIG['num_classes'])}
    recall_data = {i: [] for i in range(1, CONFIG['num_classes'])}
    f1_data = {i: [] for i in range(1, CONFIG['num_classes'])}
    
    for epoch, metrics in metrics_history.items():
        epochs.append(epoch)
        for class_id in range(1, CONFIG['num_classes']):
            if class_id in metrics:
                precision_data[class_id].append(metrics[class_id]['precision'])
                recall_data[class_id].append(metrics[class_id]['recall'])
                f1_data[class_id].append(metrics[class_id]['f1'])
            else:
                precision_data[class_id].append(0)
                recall_data[class_id].append(0)
                f1_data[class_id].append(0)
    
    # Plot Precision
    ax = axes[0, 0]
    for class_id in range(1, CONFIG['num_classes']):
        class_name = CLASS_NAMES.get(class_id - 1, f"class_{class_id}")
        ax.plot(epochs, precision_data[class_id], marker='o', label=class_name, linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision per Class', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Plot Recall
    ax = axes[0, 1]
    for class_id in range(1, CONFIG['num_classes']):
        class_name = CLASS_NAMES.get(class_id - 1, f"class_{class_id}")
        ax.plot(epochs, recall_data[class_id], marker='s', label=class_name, linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Recall', fontsize=12)
    ax.set_title('Recall per Class', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Plot F1 Score
    ax = axes[1, 0]
    for class_id in range(1, CONFIG['num_classes']):
        class_name = CLASS_NAMES.get(class_id - 1, f"class_{class_id}")
        ax.plot(epochs, f1_data[class_id], marker='^', label=class_name, linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('F1 Score per Class', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Plot mAP per class (if available)
    ax = axes[1, 1]
    ax.text(0.5, 0.5, 'mAP per class available\\nin TensorBoard and\\nvalidation output', 
           ha='center', va='center', fontsize=12, transform=ax.transAxes)
    ax.set_title('mAP per Class', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig


def plot_learning_rate_schedule(epochs, lrs, save_path=None):
    """Plot learning rate schedule."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(epochs, lrs, 'b-', linewidth=2, marker='o', markersize=4)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Learning Rate', fontsize=12)
    ax.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')  # Log scale for better visualization
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig


def save_all_charts(train_losses, val_losses, map_scores, metrics_history, 
                   charts_dir, final_epoch, learning_rates=None):
    """Save all training charts and visualizations to output directory."""
    charts_dir = Path(charts_dir)
    charts_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\\nSaving all training charts to {charts_dir}...")
    
    # 1. Training curves
    curves_path = charts_dir / "training_curves.png"
    plot_training_curves(train_losses, val_losses, map_scores, curves_path)
    print(f"  ✓ Saved: training_curves.png")
    
    # 2. Per-class metrics
    if metrics_history:
        metrics_path = charts_dir / "per_class_metrics.png"
        plot_per_class_metrics_history(metrics_history, metrics_path)
        print(f"  ✓ Saved: per_class_metrics.png")
    
    # 3. Learning rate schedule (if tracked)
    if learning_rates and len(learning_rates) > 0:
        lr_epochs = list(range(1, len(learning_rates) + 1))
        lr_path = charts_dir / "learning_rate_schedule.png"
        plot_learning_rate_schedule(lr_epochs, learning_rates, lr_path)
        print(f"  ✓ Saved: learning_rate_schedule.png")
    
    print(f"\\n✓ All charts saved successfully to {charts_dir}!")

print("✓ Comprehensive chart plotting functions ready")
print()


✓ Comprehensive chart plotting functions ready





In [20]:
# ═════════════════════════════════════════════════════════════════════════════
#                    LEARNING RATE WARMUP
# ═════════════════════════════════════════════════════════════════════════════

def get_lr_warmup(optimizer, epoch, warmup_epochs, base_lr, warmup_factor):
    """Adjust learning rate with warmup."""
    if epoch < warmup_epochs:
        # Linear warmup
        lr = base_lr * (warmup_factor + (1 - warmup_factor) * (epoch + 1) / warmup_epochs)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        return lr
    return None  # Let scheduler handle it

print("✓ Learning rate warmup function ready")
print()


✓ Learning rate warmup function ready



## Training Function


In [21]:
def train_one_epoch(model, optimizer, data_loader, device, epoch, scaler=None):
    """Train for one epoch."""
    model.train()
    
    total_loss = 0.0
    loss_dict_all = defaultdict(float)
    num_batches = 0
    
    print(f"\\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 80)
    
    start_time = time.time()
    
    # Progress bar for training batches
    pbar = tqdm(data_loader, desc=f"Training Epoch {epoch + 1}", 
                total=len(data_loader), ncols=100, unit='batch',
                bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
    
    for batch_idx, (images, targets) in enumerate(pbar):
        # Move images to device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass with mixed precision
        if scaler is not None:
            with torch.cuda.amp.autocast():
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            
            # Backward pass
            scaler.scale(losses).backward()
            
            # Gradient clipping (unscale first for AMP)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip_max_norm'])
            
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip_max_norm'])
            
            optimizer.step()
        
        # Accumulate losses
        total_loss += losses.item()
        for key, value in loss_dict.items():
            loss_dict_all[key] += value.item()
        num_batches += 1
        
        # Print diagnostic after first batch (CUDA initialization)
        if batch_idx == 0:
            print(f"\n[GPU] First batch completed - CUDA initialized")
            print(f"[GPU] Batch processing will now speed up significantly")
            if torch.cuda.is_available():
                print(f"[GPU] Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
                print(f"[GPU] Memory cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
        
        # Update progress bar with loss info
        if num_batches > 0:
            avg_loss = total_loss / num_batches
            pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    # Average losses
    avg_total_loss = total_loss / num_batches
    avg_losses = {k: v / num_batches for k, v in loss_dict_all.items()}
    
    elapsed = time.time() - start_time
    print(f"\\nEpoch {epoch + 1} Summary:")
    print(f"  Total Loss: {avg_total_loss:.4f}")
    for key, value in avg_losses.items():
        print(f"  {key}: {value:.4f}")
    print(f"  Time: {elapsed:.1f}s")
    
    # Log to TensorBoard
    if writer is not None:
        writer.add_scalar('Train/Loss', avg_total_loss, epoch)
        writer.add_scalar('Train/LearningRate', optimizer.param_groups[0]['lr'], epoch)
        for key, value in avg_losses.items():
            writer.add_scalar(f'Train/{key}', value, epoch)
    
    return avg_total_loss, avg_losses


def evaluate(model, data_loader, device, epoch=0, gradcam=None, dataset=None):
    """
    Enhanced evaluation with mAP, Grad-CAM, and TensorBoard logging.
    """
    model.eval()
    
    total_loss = 0.0
    loss_dict_all = defaultdict(float)
    num_batches = 0
    
    # For mAP calculation
    all_predictions = []
    all_targets = []
    
    # For Grad-CAM visualization
    gradcam_samples = []
    
    print("\\nValidation...")
    
    with torch.no_grad():
        # Progress bar for validation batches
        pbar = tqdm(data_loader, desc=f"Validation", 
                    total=len(data_loader), ncols=100, unit='batch',
                    bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
        
        for batch_idx, (images, targets) in enumerate(pbar):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Forward pass - get both loss and predictions
            # Note: In eval mode, model(images, targets) returns loss dict
            #       model(images) returns predictions
            if CONFIG['use_amp']:
                with torch.cuda.amp.autocast():
                    loss_dict = model(images, targets)
                    losses = sum(loss for loss in loss_dict.values())
            else:
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            
            # Get predictions separately (model in eval mode)
            with torch.no_grad():
                if CONFIG['use_amp']:
                    with torch.cuda.amp.autocast():
                        predictions = model(images)
                else:
                    predictions = model(images)
            
            total_loss += losses.item()
            for key, value in loss_dict.items():
                loss_dict_all[key] += value.item()
            num_batches += 1
            
            # Collect predictions and targets for mAP
            if CONFIG['compute_map']:
                all_predictions.extend(predictions)
                all_targets.extend(targets)
            
            # Collect samples for Grad-CAM (first batch only, limited samples)
            if CONFIG['gradcam_enabled'] and gradcam is not None and batch_idx == 0:
                for i in range(min(CONFIG['gradcam_samples'], len(images))):
                    if len(predictions[i]['boxes']) > 0:  # Only if detections exist
                        gradcam_samples.append({
                            'image': images[i],
                            'target': targets[i],
                            'prediction': predictions[i]
                        })
    
    # Calculate average losses
    avg_total_loss = total_loss / num_batches
    avg_losses = {k: v / num_batches for k, v in loss_dict_all.items()}
    
    print(f"Validation Loss: {avg_total_loss:.4f}")
    for key, value in avg_losses.items():
        print(f"  {key}: {value:.4f}")
    
    # Calculate mAP
    map_score = 0.0
    ap_per_class = {}
    if CONFIG['compute_map'] and len(all_predictions) > 0:
        print("\\nCalculating mAP...")
        map_score, ap_per_class = calculate_map(all_predictions, all_targets)
        print(f"mAP@0.5: {map_score:.4f}")
        for class_id, ap in ap_per_class.items():
            class_name = CLASS_NAMES.get(class_id - 1, f"class_{class_id}")
            print(f"  {class_name}: {ap:.4f}")
    
    # Generate Grad-CAM visualizations
    if CONFIG['gradcam_enabled'] and gradcam is not None and len(gradcam_samples) > 0:
        print(f"\\nGenerating Grad-CAM visualizations ({len(gradcam_samples)} samples)...")
        for idx, sample in enumerate(gradcam_samples):
            # Get original image from dataset
            if dataset is not None:
                # Find image in dataset (this is a simplified approach)
                # In practice, you'd want to track image indices
                try:
                    # Convert tensor image back to PIL for visualization
                    img_tensor = sample['image'].cpu()
                    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype('uint8')
                    from PIL import Image as PILImage
                    pil_image = PILImage.fromarray(img_np)
                    
                    save_path = CONFIG['gradcam_save_dir'] / f"epoch_{epoch}_sample_{idx}.png"
                    
                    # Actually generate and save Grad-CAM
                    gradcam.visualize(sample['image'], pil_image, save_path=save_path)
                    print(f"  ✓ Saved Grad-CAM: {save_path.name}")
                except Exception as e:
                    print(f"  ⚠ Warning: Could not save Grad-CAM: {e}")
    
    # Log to TensorBoard
    if writer is not None:
        writer.add_scalar('Validation/Loss', avg_total_loss, epoch)
        for key, value in avg_losses.items():
            writer.add_scalar(f'Validation/{key}', value, epoch)
        if map_score > 0:
            writer.add_scalar('Validation/mAP', map_score, epoch)
            for class_id, ap in ap_per_class.items():
                class_name = CLASS_NAMES.get(class_id - 1, f"class_{class_id}")
                writer.add_scalar(f'Validation/AP_{class_name}', ap, epoch)
    
    return avg_total_loss, avg_losses, map_score, ap_per_class

print("✓ Training and evaluation functions ready")
print()


✓ Training and evaluation functions ready



## Checkpoint Management


In [22]:
def save_checkpoint(model, optimizer, lr_scheduler, epoch, loss, save_dir, is_best=False):
    """Save training checkpoint."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Get model state (handle DataParallel)
    model_state = model.module.state_dict() if isinstance(model, (DataParallel, DistributedDataParallel)) else model.state_dict()
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'lr_scheduler_state_dict': lr_scheduler.state_dict(),
        'loss': loss,
        'config': CONFIG,
    }
    
    # Save regular checkpoint
    checkpoint_path = save_dir / f"checkpoint_epoch_{epoch + 1}.pth"
    torch.save(checkpoint, checkpoint_path)
    
    # Save best model
    if is_best:
        best_path = save_dir / "best_model.pth"
        torch.save(checkpoint, best_path)
        print(f"✓ Saved best model to {best_path}")
    
    # Save latest checkpoint
    latest_path = save_dir / "latest_checkpoint.pth"
    torch.save(checkpoint, latest_path)
    
    print(f"✓ Saved checkpoint to {checkpoint_path}")


def load_checkpoint(model, optimizer, lr_scheduler, checkpoint_path, device):
    """Load training checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state
    if isinstance(model, (DataParallel, DistributedDataParallel)):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
    
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    lr_scheduler.load_state_dict(checkpoint['lr_scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    
    print(f"✓ Loaded checkpoint from {checkpoint_path}")
    print(f"  Resuming from epoch {start_epoch}")
    
    return start_epoch

print("✓ Checkpoint functions ready")
print()


✓ Checkpoint functions ready



## Main Training Loop


## Optional: Learning Rate Finder

Run this cell BEFORE training to find optimal learning rate automatically.


In [23]:
# ═════════════════════════════════════════════════════════════════════════════
#                    OPTIONAL: FIND OPTIMAL LEARNING RATE
# ═════════════════════════════════════════════════════════════════════════════

# Uncomment to run LR finder (recommended before first training)
# This will test different learning rates and recommend the best one
# 
# recommended_lr, lrs, losses = find_learning_rate(model, train_loader, device)
# 
# # Update CONFIG with recommended LR (optional)
# # CONFIG['learning_rate'] = recommended_lr * 0.3  # Use 30% of recommended for stability
# # print(f"\\nUpdated learning rate in CONFIG to: {CONFIG['learning_rate']}")

print("LR Finder ready (uncomment to use)")
print()


LR Finder ready (uncomment to use)



In [24]:
# ═════════════════════════════════════════════════════════════════════════════
#                    EXPORT MODEL FOR DEPLOYMENT (ACTIVE)
# ═════════════════════════════════════════════════════════════════════════════

# Export model after training completes
# This will automatically export the best model to ONNX format

# Load best model (only if it exists)
best_model_path = CONFIG['save_dir'] / "best_model.pth"
if not best_model_path.exists():
    print("⚠ best_model.pth not found. Train the model first!")
    print(f"  Expected path: {best_model_path}")
else:
    checkpoint = torch.load(best_model_path, map_location=device)
    if isinstance(model, (DataParallel, DistributedDataParallel)):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
    
    # Export to ONNX (recommended for deployment)
    export_dir = CONFIG['save_dir'] / "exported"
    export_model(model, export_dir, format='onnx')
    
    # Or export to TorchScript (alternative)
    # export_model(model, export_dir, format='torchscript')
    
    print("✓ Model export code active - run this cell after training completes")
    print()


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\fyp_data\\mask_rcnn_runs\\best_model.pth'

## Optional: Export Model After Training

Export trained model to ONNX or TorchScript for deployment.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#                    OPTIONAL: EXPORT MODEL FOR DEPLOYMENT
# ═════════════════════════════════════════════════════════════════════════════

# Uncomment after training to export model
# 
# # Load best model
# checkpoint = torch.load(CONFIG['save_dir'] / "best_model.pth", map_location=device)
# if isinstance(model, (DataParallel, DistributedDataParallel)):
#     model.module.load_state_dict(checkpoint['model_state_dict'])
# else:
#     model.load_state_dict(checkpoint['model_state_dict'])
# 
# # Export to ONNX (recommended for deployment)
# export_dir = CONFIG['save_dir'] / "exported"
# export_model(model, export_dir, format='onnx')
# 
# # Or export to TorchScript
# # export_model(model, export_dir, format='torchscript')

print("Model export ready (uncomment after training)")
print()


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#                            MAIN TRAINING LOOP
# ═════════════════════════════════════════════════════════════════════════════

def main_training_loop():
    """Enhanced training loop with early stopping, warmup, Grad-CAM, and TensorBoard."""
    
    print("="*100)
    print(" Starting Enhanced Training ".center(100, "="))
    print("="*100)
    print()
    
    # Initialize Grad-CAM if enabled
    gradcam = None
    if CONFIG['gradcam_enabled']:
        try:
            # Get model without DataParallel wrapper for Grad-CAM
            model_for_gradcam = model.module if isinstance(model, (DataParallel, DistributedDataParallel)) else model
            gradcam = GradCAM(model_for_gradcam)
            print("✓ Grad-CAM initialized")
        except Exception as e:
            print(f"⚠ Could not initialize Grad-CAM: {e}")
            gradcam = None
    
    # Resume from checkpoint if specified
    start_epoch = 0
    best_val_loss = float('inf')
    best_map = 0.0
    
    if CONFIG['resume_from'] and Path(CONFIG['resume_from']).exists():
        start_epoch = load_checkpoint(model, optimizer, lr_scheduler, CONFIG['resume_from'], device)
    
    # Early stopping
    early_stopping_counter = 0
    patience = CONFIG['early_stopping_patience'] if CONFIG['early_stopping'] else float('inf')
    
    # Training history
    train_losses = []
    val_losses = []
    map_scores = []
    
    # Training loop
    for epoch in range(start_epoch, CONFIG['epochs']):
        print(f"\\n{'='*100}")
        print(f" EPOCH {epoch + 1}/{CONFIG['epochs']} ".center(100, "="))
        print(f"{'='*100}\\n")
        
        # Learning rate warmup
        warmup_lr = get_lr_warmup(
            optimizer, epoch, 
            CONFIG['warmup_epochs'], 
            CONFIG['learning_rate'],
            CONFIG['warmup_factor']
        )
        if warmup_lr is not None:
            print(f"Warmup LR: {warmup_lr:.6f}")
        else:
            # Normal scheduler step
            lr_scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Learning rate: {current_lr:.6f}")
        
        # Train
        train_loss, train_losses_dict = train_one_epoch(
            model, optimizer, train_loader, device, epoch, scaler
        )
        train_losses.append(train_loss)
        
        # Validate
        is_best = False
        val_loss = None
        map_score = 0.0
        ap_per_class = {}
        
        if (epoch + 1) % CONFIG['val_period'] == 0:
            val_loss, val_losses_dict, map_score, ap_per_class = evaluate(
                model, val_loader, device, epoch, gradcam, val_dataset
            )
            val_losses.append(val_loss)
            map_scores.append(map_score)
            
            # Check if best model (use mAP if available, else validation loss)
            if map_score > 0:
                is_best = map_score > best_map
                if is_best:
                    best_map = map_score
                    best_val_loss = val_loss
                    print(f"\\n✓ New best mAP: {best_map:.4f}")
            else:
                is_best = val_loss < best_val_loss
                if is_best:
                    best_val_loss = val_loss
                    print(f"\\n✓ New best validation loss: {best_val_loss:.4f}")
            
            # Early stopping check
            if CONFIG['early_stopping']:
                if is_best:
                    early_stopping_counter = 0
                else:
                    early_stopping_counter += 1
                    print(f"Early stopping counter: {early_stopping_counter}/{patience}")
                    
                    if early_stopping_counter >= patience:
                        print(f"\\n{'='*100}")
                        print(" Early Stopping Triggered ".center(100, "="))
                        print(f"{'='*100}\\n")
                        print(f"No improvement for {patience} epochs. Stopping training.")
                        break
        
        # Save checkpoint
        if (epoch + 1) % CONFIG['save_period'] == 0:
            save_checkpoint(
                model, optimizer, lr_scheduler, epoch, train_loss,
                CONFIG['save_dir'], is_best=is_best
            )
    
    # Final checkpoint
    print(f"\\n{'='*100}")
    print(" Training Complete ".center(100, "="))
    print(f"{'='*100}\\n")
    save_checkpoint(model, optimizer, lr_scheduler, epoch, train_loss, CONFIG['save_dir'])
    
    # Close TensorBoard writer
    if writer is not None:
        writer.close()
    
    print(f"\\nTraining finished!")
    print(f"Best validation loss: {best_val_loss:.4f}")
    if best_map > 0:
        print(f"Best mAP: {best_map:.4f}")
    print(f"Checkpoints saved to: {CONFIG['save_dir']}")
    if CONFIG['tensorboard_enabled']:
        print(f"TensorBoard logs: {CONFIG['tensorboard_log_dir']}")
    if CONFIG['gradcam_enabled']:
        print(f"Grad-CAM visualizations: {CONFIG['gradcam_save_dir']}")
    
    return train_losses, val_losses, map_scores


# Start training
train_losses, val_losses, map_scores = main_training_loop()

print("✓ Training loop ready")
print("  Training starts automatically when this cell runs")
print()


✓ Training loop ready
  Uncomment the last line to start training



## Start Training

**Note**: Make sure the dataset creation is complete before running this cell.  
The dataset should be in `final_balanced` directory with proper train/val/test splits.


In [ ]:
# Check if dataset is ready
print("="*100)
print(" CHECKING DATASET & STARTING TRAINING ".center(100, "="))
print("="*100)
print()

if DATASET_DIR.exists() and (DATASET_DIR / "images" / "train").exists():
    print(" Checking dataset files...")
    train_files = []
    train_dir = DATASET_DIR / "images" / "train"
    for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
        files = list(train_dir.glob(f"*{ext}"))
        if files:
            print(f"  Found {len(files)} {ext} files")
        train_files.extend(files)
    
    print()
    if len(train_files) > 0:
        print(f"OK Dataset ready: {len(train_files)} training images found")
        print(f" Total batches per epoch: {len(train_loader)}")
        print(f" Validation every {CONFIG['val_period']} epochs")
        print(f" Checkpoints every {CONFIG['save_period']} epochs")
        print()
        print(" Starting enhanced training with:")
        print("  * Multi-GPU training (3x RTX A2000)")
        print("  * Grad-CAM visualization")
        print("  * mAP calculation")
        print("  * TensorBoard logging")
        print("  * Early stopping (patience: 15)")
        print("  * Learning rate warmup (5 epochs)")
        print("  * Gradient clipping (max norm: 10.0)")
        print("  * Mixed precision training")
        print()
        print(f" Training for {CONFIG['epochs']} epochs (or until early stopping)")
        print("  First batch can take ~2-5 minutes while CUDA/data workers warm up")
        print("  Subsequent batches run in a few seconds")
        print()
        print("="*100)
        
        # Start training
        import time
        training_start = time.time()
        train_losses, val_losses, map_scores = main_training_loop()
        training_elapsed = time.time() - training_start
        
        # Training complete summary
        print()
        print("="*100)
        print(" TRAINING COMPLETED SUCCESSFULLY! ".center(100, "="))
        print("="*100)
        print()
        print(f"  Total training time: {training_elapsed/3600:.2f} hours ({training_elapsed/60:.1f} minutes)")
        print(f" Epochs completed: {len(train_losses)}/{CONFIG['epochs']}")
        print(f" Final training loss: {train_losses[-1]:.4f}")
        if val_losses:
            print(f" Final validation loss: {val_losses[-1]:.4f}")
        if map_scores:
            print(f" Final mAP: {map_scores[-1]:.4f}")
            print(f" Best mAP: {max(map_scores):.4f}")
        print()
        print(" Output files saved to:")
        print(f"  • Checkpoints: {CONFIG['save_dir']}")
        print(f"  • TensorBoard: {CONFIG['tensorboard_log_dir']}")
        print(f"  • Grad-CAM: {CONFIG['gradcam_save_dir']}")
        print()
        print("OK You can now run the export cell to export the best model!")
        print("="*100)
    else:
        print("WARNING:  Dataset directory exists but no images found")
        print("   Waiting for dataset creation to complete...")
else:
    print("WARNING:  Dataset directory not ready yet")
    print(f"   Expected path: {DATASET_DIR}")
    print("   Waiting for dataset creation to complete...")


✓ Dataset ready: 31918 training images found
\nStarting enhanced training with:
  • Grad-CAM visualization
  • mAP calculation
  • TensorBoard logging
  • Early stopping
  • Learning rate warmup
==================================== Starting Enhanced Training ====================================

✓ Grad-CAM initialized
\n====================================================================================================
=========================================== EPOCH 1/100 ============================================
====================================================================================================\n
Warmup LR: 0.001400
Learning rate: 0.001400
\nEpoch 1/100
--------------------------------------------------------------------------------


## Verification: Check Label Conversion

Verify that YOLO labels are correctly converted to Mask R-CNN format.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#                    VERIFICATION: LABEL CONVERSION
# ═════════════════════════════════════════════════════════════════════════════

def verify_label_conversion():
    """Verify YOLO to Mask R-CNN label conversion is correct."""
    
    print("="*100)
    print(" Label Conversion Verification ".center(100, "="))
    print("="*100)
    print()
    
    # Class mapping
    print("Class Mapping:")
    print("  YOLO Dataset -> Mask R-CNN Model")
    yolo_classes = {0: 'anus', 1: 'breast', 2: 'female_genital', 3: 'male_genital'}
    for yolo_id, name in yolo_classes.items():
        maskrcnn_id = yolo_id + 1
        print(f"    {yolo_id} ({name:15s}) -> {maskrcnn_id} ({name})")
    print(f"    -  (background)       -> 0 (background, Mask R-CNN reserved)")
    
    print()
    print("="*100)
    
    # Check if dataset exists
    if not DATASET_DIR.exists():
        print("⚠ Dataset not ready yet. Skipping verification.")
        return
    
    # Load a sample
    try:
        temp_dataset = YOLOToMaskRCNNDataset(DATASET_DIR, split='train')
        
        if len(temp_dataset) == 0:
            print("⚠ No images found in dataset.")
            return
        
        print(f"✓ Dataset loaded: {len(temp_dataset)} images in train split")
        
        # Test first few samples
        print("\\nTesting first 3 samples:")
        for idx in range(min(3, len(temp_dataset))):
            image, target = temp_dataset[idx]
            
            print(f"\\n  Sample {idx + 1}:")
            print(f"    Image shape: {image.shape}")
            print(f"    Number of objects: {len(target['boxes'])}")
            
            if len(target['boxes']) > 0:
                for i in range(min(3, len(target['boxes']))):
                    box = target['boxes'][i]
                    label = target['labels'][i].item()
                    class_name = CLASS_NAMES.get(label - 1, 'unknown')  # -1 to get back YOLO id
                    
                    print(f"      Object {i+1}: class={label} ({class_name}), "
                          f"box=[{box[0]:.1f}, {box[1]:.1f}, {box[2]:.1f}, {box[3]:.1f}]")
                    
                    # Validate box coordinates
                    if box[0] >= box[2] or box[1] >= box[3]:
                        print(f"        ⚠ WARNING: Invalid box coordinates!")
                    if label < 1 or label > 4:
                        print(f"        ⚠ WARNING: Invalid class label!")
        
        print("\\n" + "="*100)
        print(" Verification Complete ".center(100, "="))
        print("="*100)
        print()
        print("✓ Label conversion verified successfully")
        print("✓ YOLO polygon segmentation -> Mask R-CNN bounding boxes")
        print("✓ Class IDs correctly mapped (+1 offset for background)")
        
    except Exception as e:
        print(f"⚠ Error during verification: {e}")
        import traceback
        traceback.print_exc()


# Run verification
verify_label_conversion()
